In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
#for f in *.sh; do sbatch "$f"; done
#merges multiple batches into one file per L value
#
Larr = np.logspace(1,4.3,20)[:18]
colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
labels=[r"Percolation Probability in X $\Pi_X$",r"Percolation Probability in Y $\Pi_Y$",r"Percolation Probability in X or Y $\Pi_{XY}$" ]
for k in range(3):
    plt.figure(figsize=(8,5))
    for j in range(len(Larr)):
        # print(j)
        L = int(Larr[j])
        skip = max(1,int(L*L/1000))
        estimate_seconds_per_run = 5*((L/1000)**2)
        runs = 10*int(1*60*60/estimate_seconds_per_run) #how many runs to do total
        file_path = f"postprocess/postprocessedquantities_L_{L}_runs_{runs}.npy"  # Path to your file
        arr = np.load(file_path, mmap_mode="r")
        plt.plot(np.arange(0,L*L+1,skip)/(L*L),arr[0,::skip,k],label=f"L={L}",color=colors[j])
    plt.xlabel("n/N")
    plt.legend()
    plt.ylabel(labels[k])
    plt.title(labels[k])
    plt.show() 
for k in range(3):
    plt.figure(figsize=(8,5))
    for j in range(len(Larr)):
        # print(j)
        L = int(Larr[j])
        skip = max(1,int(L*L/1000))
        estimate_seconds_per_run = 5*((L/1000)**2)
        runs = 10*int(1*60*60/estimate_seconds_per_run) #how many runs to do total
        file_path = f"postprocess/postprocessedquantities_L_{L}_runs_{runs}.npy"  # Path to your file
        arr = np.load(file_path, mmap_mode="r")
        plt.plot(np.arange(0,L*L+1,skip)/(L*L),arr[1,::skip,k]/arr[0,::skip,k],label=f"L={L}",color=colors[j])
    plt.xlabel("n/N")
    plt.legend()
    plt.ylabel("Fractional Errors in "+labels[k])
    plt.title("Fractional Errors in "+labels[k])
    plt.yscale("log")
    plt.show() 

In [ ]:
from scipy.stats import binom

def nz_binomial_transform(Qn, Qnerr, pvals, nsigma=5):
    N = len(Qn)
    Qp = np.zeros(len(pvals))
    Qperr = np.zeros(len(pvals))

    for i in range(len(pvals)):
        p = pvals[i]
        if p == 0:
            Qp[i] = Qn[0]
            continue
        if p == 1:
            Qp[i] = Qn[N-1]
            continue
        mean = N * p  
        sigma = np.sqrt(N * p * (1 - p))
        nmin = max(0, int(mean - nsigma * sigma)-1)#Avoid summing over entire dataset, only 8 sigma closest to mean
        nmax = min(N, int(mean + nsigma * sigma) )#remember python indexing, mean is at index mean-1
        n = np.arange(nmin, nmax)
        weights = binom.pmf(n+1, N, p)
        Qp[i] = np.nansum(Qn[n] * weights)
        Qperr[i] = np.sqrt(np.nansum(np.square(Qnerr[n]*weights)))
    return Qp,Qperr

import numpy as np 
import matplotlib.pyplot as plt
#for f in *.sh; do sbatch "$f"; done
#merges multiple batches into one file per L value
#
pvals = np.linspace(0,1,1000)
Larr = np.logspace(1,4.3,20)[:18]
colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
labels=[r"Percolation Probability in X $\Pi_X$",r"Percolation Probability in Y $\Pi_Y$",r"Percolation Probability in X or Y $\Pi_{XY}$" ]

for k in range(3):
    fig1,ax1 = plt.subplots(figsize=(8,5))
    fig2,ax2 = plt.subplots(figsize=(8,5))
    for j in range(len(Larr)):
        print(j)
        L = int(Larr[j])
        skip = max(1,int(L*L/1000))
        estimate_seconds_per_run = 5*((L/1000)**2)
        runs = 10*int(1*60*60/estimate_seconds_per_run) #how many runs to do total
        file_path = f"postprocess/postprocessedquantities_L_{L}_runs_{runs}.npy"  # Path to your file
        arr = np.load(file_path, mmap_mode="r")
        qp,qperr = nz_binomial_transform(arr[0,:,k], arr[1,:,k], pvals, nsigma=8)
        ax1.plot(pvals,qp,label=f"L={L}",color=colors[j])
        ax2.plot(pvals,qperr/qp,label=f"L={L}",color=colors[j])

    ax1.set_xlabel(r"Occupation Probability $p$")
    ax2.set_xlabel(r"Occupation Probability $p$")

    ax1.legend()
    ax1.set_ylabel(labels[k])
    ax2.set_ylabel("Fractional Errors in "+labels[k])
    ax2.set_yscale("log")
    ax1.set_title(labels[k])
    ax2.set_title("Fractional Errors in "+labels[k])
    plt.show()
    plt.show()

In [ ]:
#crossing point
from scipy.optimize import curve_fit
from scipy.stats import binom
import matplotlib.pyplot as plt
import numpy as np
Larr = np.logspace(1,4.3,20)[:18]
datalist=[]
def nz_binomial_transform(Qn, Qnerr, pvals, nsigma=5):
    N = len(Qn)
    Qp = np.zeros(len(pvals))
    Qperr = np.zeros(len(pvals))

    for i in range(len(pvals)):
        p = pvals[i]
        if p == 0:
            Qp[i] = Qn[0]
            continue
        if p == 1:
            Qp[i] = Qn[N-1]
            continue
        mean = N * p  
        sigma = np.sqrt(N * p * (1 - p))
        nmin = max(0, int(mean - nsigma * sigma)-1)#Avoid summing over entire dataset, only 8 sigma closest to mean
        nmax = min(N, int(mean + nsigma * sigma) )#remember python indexing, mean is at index mean-1
        n = np.arange(nmin, nmax)
        weights = binom.pmf(n+1, N, p)
        Qp[i] = np.nansum(Qn[n] * weights)
        Qperr[i] = np.sqrt(np.nansum(np.square(Qnerr[n]*weights)))
    return Qp,Qperr
for j in range(len(Larr)):
    print(j)
    L = int(Larr[j])
    skip = max(1,int(L*L/1000))
    estimate_seconds_per_run = 5*((L/1000)**2)
    runs = 10*int(1*60*60/estimate_seconds_per_run) #how many runs to do total
    file_path = f"postprocess/postprocessedquantities_L_{L}_runs_{runs}.npy"  # Path to your file
    arr = np.load(file_path, mmap_mode="r")
    datalist.append(arr[:,:,0])
def findcrossing(xarr,p):
    residuals = np.zeros(xarr.shape[0])
    for i in range(len(datalist)):
        arr=nz_binomial_transform(datalist[i][0,:], datalist[i][1,:], np.array([p.item()]), nsigma=5)
        residuals[i]=arr[0]
        print(arr[0])
    return residuals-np.mean(residuals)
popt,pcov = curve_fit(findcrossing,np.zeros(len(Larr)),np.zeros(len(Larr)), p0=[0.59])
print(f"{popt} +=- {np.sqrt(pcov)}")
xarr = np.arange(18)
plt.scatter(xarr,findcrossing(xarr,*popt))

In [ ]:
#max slope
#crossing point
from scipy.optimize import curve_fit

Larr = np.logspace(1,4.3,20)[:18]
datalist=[]
for j in range(len(Larr)):
    print(j)
    L = int(Larr[j])
    skip = max(1,int(L*L/1000))
    estimate_seconds_per_run = 5*((L/1000)**2)
    runs = 10*int(1*60*60/estimate_seconds_per_run) #how many runs to do total
    file_path = f"postprocess/postprocessedquantities_L_{L}_runs_{runs}.npy"  # Path to your file
    arr = np.load(file_path, mmap_mode="r")
    datalist.append(arr[:,:,2])
def maxslope(xarr,p):
    dp=1e-3
    residuals = np.zeros(xarr.shape[0])
    for i in range(len(datalist)):
        arr,qperr=nz_binomial_transform(datalist[i][0,:], datalist[i][1,:], np.array([p.item()+dp,p.item()-dp]), nsigma=5)
        
        residuals[i]=(arr[0]-arr[1])/(2*dp)
        print(arr[0])
    print(residuals)
    return 1/residuals
popt,pcov = curve_fit(maxslope,np.zeros(len(Larr)),np.zeros(len(Larr)), p0=[0.59],bounds=([0.3],[0.7]))
print(f"{popt} +=- {np.sqrt(pcov)}")

In [ ]:
#finite size scaling
# finite size scaling
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
#Larr
#Order parameter in Warr[L,p]
#pvals
datalist=[]
for j in range(len(Larr)):
    print(j)
    L = int(Larr[j])
    skip = max(1,int(L*L/1000))
    estimate_seconds_per_run = 5*((L/1000)**2)
    runs = 10*int(1*60*60/estimate_seconds_per_run) #how many runs to do total
    file_path = f"postprocess/postprocessedquantities_L_{L}_runs_{runs}.npy"  # Path to your file
    arr = np.load(file_path, mmap_mode="r")
    datalist.append(arr[:,:,2])

xrange  = 2
xbase = np.linspace(-xrange,xrange,1000)
pvals = np.linspace(0,1,1000)
Warr = np.zeros((len(datalist),pvals.shape[0]))
Warrerr = np.zeros((len(datalist),pvals.shape[0]))

for i in range(len(Larr)):
    arr,qperr=nz_binomial_transform(datalist[i][0,:], datalist[i][1,:], pvals, nsigma=5)
    Warr[i,:] = arr
    Warrerr[i,:] = qperr

def finite_size_scaling_residuals(xbase,pc,nu):
    mean = np.zeros(len(xbase))
    residuals = np.zeros(len(xbase))
    for i in range(len(Larr)):
        L = Larr[i]
        x = (pvals-pc)*L**(1/nu)
        y = Warr[i,:]
        f = interp1d(
            x,
            y,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        # print(x)
        # print(y)
        regridy = f(xbase)
        mean += regridy
    mean /= len(Larr)
    for i in range(len(Larr)):
        L = Larr[i]
        x = (pvals-pc)*L**(1/nu)
        y = Warr[i,:]
        f = interp1d(
            x,
            y,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        regridy = f(xbase)
    residuals += np.square(mean-regridy)
    return residuals
p0 = [0.5926,4/3]
popt,pcov = curve_fit(finite_size_scaling_residuals,xbase,np.zeros(xbase.shape[0]),p0=p0)
pc,nu =popt
perr = np.sqrt(np.diag(pcov))
print(f"pc = {pc} +- {perr[0]} expect {p0[0]}")
print(f"nu = {nu} +- {perr[1]} expect {p0[1]}")
for i in range(len(Larr)):
    print(i)
    x = (pvals-pc)*Larr[i]**(1/nu)
    y = Warr[i,:]
    plt.plot(x,y,label=f"L={Larr[i]}")
plt.xlim(np.min(xbase),np.max(xbase))
plt.xlabel(r"$(p_i-p_c)L^{1/\nu}$")
plt.ylabel(r"$\Pi_X$")
plt.title(r"Data Collapse of Percolation Probability $\Pi_X$")
plt.show()
plt.title("Variance of Best Fit Data Collapse From Mean")
plt.plot(xbase,finite_size_scaling_residuals(xbase,*popt))
plt.xlabel(r"$(p_i-p_c)L^{1/\nu}$")
plt.show()